# ExtrusionExtraction: interactive Colab reconstruction (experimental)
Run the cells in your own Google runtime. CPU only; no paid upgrade or hosted backend. Free runtime availability is not guaranteed. The precompiled worker includes Java, ONNX and SkinTokens; model weights are downloaded separately and verified by the worker. References remain in your temporary runtime. Export before disconnecting. [Colab FAQ](https://research.google.com/colaboratory/faq.html).


In [ ]:
from pathlib import Path
import hashlib, json, os, re, subprocess, tarfile, tempfile, urllib.request
root = Path(tempfile.mkdtemp(prefix='ee-'))
# Optional exact four-part version, e.g. v0.1.2.3. Empty selects newest published compatible worker.
VERSION = ''
pattern = re.compile(r'ExtrusionExtraction-(v[0-9]+\.[0-9]+\.[0-9]+\.[0-9]+)-colab-linux-x86_64\.tar\.gz')
candidates = []
for page in range(1, 11):
    url = f'https://api.github.com/repos/HereLiesAz/ExtrusionExtraction/releases?per_page=100&page={page}'
    with urllib.request.urlopen(url, timeout=30) as response:
        releases = json.load(response)
    if not releases:
        break
    for release in releases:
        if release['draft']:
            continue
        for asset in release['assets']:
            match = pattern.fullmatch(asset['name'])
            if match and (not VERSION or match[1] == VERSION):
                candidates.append((tuple(map(int, match[1][1:].split('.'))), asset))
assert candidates, 'No published Colab worker found. A successful release must publish one first.'
version, asset = max(candidates, key=lambda entry: entry[0])
digest = asset.get('digest') or ''
assert re.fullmatch(r'sha256:[0-9a-f]{64}', digest), 'Release asset lacks SHA-256 metadata.'
assert 0 < asset['size'] <= 512 * 1024 * 1024, 'Invalid worker archive size.'
download_url = asset['browser_download_url']
assert download_url.startswith('https://github.com/HereLiesAz/ExtrusionExtraction/releases/download/'), 'Unexpected release URL.'
archive = root / 'worker.tar.gz'
sha = hashlib.sha256()
received = 0
with urllib.request.urlopen(download_url, timeout=60) as source, archive.open('wb') as target:
    while chunk := source.read(1024 * 1024):
        received += len(chunk)
        assert received <= asset['size'], 'Worker exceeds expected size.'
        sha.update(chunk)
        target.write(chunk)
assert received == asset['size'] and 'sha256:' + sha.hexdigest() == digest, 'Worker integrity check failed.'
# Fail closed if the runtime lacks safe tar extraction.
assert hasattr(tarfile, 'data_filter'), 'Safe tar extraction is unavailable in this runtime.'
with tarfile.open(archive, 'r:gz') as package:
    members = package.getmembers()
    assert all(m.name == 'worker' or m.name.startswith('worker/') for m in members)
    assert sum(m.size for m in members) <= 1024 * 1024 * 1024, 'Expanded worker exceeds size limit.'
    package.extractall(root, members=members, filter='data')
worker = root / 'worker'
archive.unlink()
print('Worker version:', '.'.join(map(str, version)))
print('Source revision:', (worker / 'source-revision.txt').read_text().strip())


## Runtime check
Checks the actual packaged Java, ONNX and SkinTokens runtime before uploading any references. No source build or Android SDK. Linux CPU and native ABI compatibility are required.


In [ ]:
subprocess.run([str(worker / 'run'), '--check'], check=True)


In [ ]:
from google.colab import files
# Upload one .eejob exported by the app, or 4–24 individual reference images.
upload_dir = Path(tempfile.mkdtemp(prefix='upload-', dir=root))
previous = Path.cwd()
try:
    os.chdir(upload_dir)
    uploaded = files.upload()
finally:
    os.chdir(previous)
print(list(uploaded))


In [ ]:
import struct
ORDER = [] # only needed for individual images: exact filenames in viewing order
CELLS = 72
MASK = True
RIG = 'both' # none, template, learned, both; ignored for .eejob
refs = Path(tempfile.mkdtemp(prefix='refs-', dir=root))
output = root / ('output-' + refs.name)
command = [str(worker / 'run')]
jobs = [name for name in uploaded if name.lower().endswith('.eejob')]
if jobs:
    assert len(jobs) == 1 and len(uploaded) == 1, 'Upload exactly one job file.'
    data = uploaded[jobs[0]]
    assert len(data) >= 16, 'Truncated job.'
    magic, cells, flags, count = struct.unpack_from('>4I', data)
    assert magic == 0x45454a31 and 32 <= cells <= 96 and 0 <= flags <= 7 and 4 <= count <= 24
    assert len(data) == 16 + count * 504 * 504 * 4, 'Invalid job length.'
    path = refs / 'reconstruction.eejob'
    path.write_bytes(data)
    args = ['--job', str(root/'models'), str(output), str(path)]
    needs_native = bool(flags & 4)
    print(f'{count} references, {cells} cells, mask={bool(flags & 1)}, template={bool(flags & 2)}, learned={needs_native}')
else:
    assert 4 <= len(ORDER) <= 24 and len(set(ORDER)) == len(ORDER) and set(ORDER) == set(uploaded)
    assert 32 <= CELLS <= 96 and RIG in ('none', 'template', 'learned', 'both')
    paths = []
    for i, name in enumerate(ORDER):
        path = refs / f'{i:02d}.image'
        path.write_bytes(uploaded[name])
        paths.append(str(path))
    args = [str(root/'models'), str(output), str(CELLS), 'mask' if MASK else 'no-mask', RIG, *paths]
status = subprocess.run(command + args).returncode
print('Exit status:', status)
print('Successful exports:', [p.name for p in output.glob('*.glb')])
if status: print('At least one stage failed; inspect the log. Successful prior variants remain downloadable.')


In [ ]:
# Download completed files even if an optional rig failed.
exports = sorted(output.glob('*.glb'))
assert exports, 'No completed GLBs.'
for export in exports:
    files.download(str(export))
